# Volume 5, Chapter 8: Sieves (Python companion)

Exercises first, solutions at the end. We run Legendre's inclusion–exclusion and watch the factor $2e^{-\gamma}$ appear (Proposition 8.1, Exercise 8.2), compute Selberg's weights and check that they are at most $1$ (Theorem 8.2), count twin primes and estimate Brun's constant (Theorem 8.5, Exercise 8.6), test the large sieve on the squares (Proposition 8.11) and on random coefficients (Theorem 8.8), and end with the cost of sieving on a computer. Requirements: numpy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P8.1 (★) Legendre.** Implement Legendre's identity by brute-force inclusion–exclusion and compare with a direct count. Then compute $x\prod_{p\le\sqrt x}(1-\frac1p)$ divided by $\Phi(x,\sqrt x)$ for $x=10^3,\dots,10^7$.

**P8.2 (★★) Selberg's weights.** Compute the optimal $\lambda_d$ for the integers ($g(d)=\frac1d$) with $y=60$, check $|\lambda_d|\le1$ and $G(y)\ge\log y$ (Lemma 8.3), and use them to bound the number of primes in $]10^9,10^9+10^5]$.

**P8.3 (★) Twins.** Count twin primes up to $10^7$, compare with $2C_2\int_2^x\frac{dt}{\log^2t}$, and estimate Brun's constant by the extrapolation of Exercise 8.6.

**P8.4 (★★) The large sieve.** Find the $n\le10^6$ that are squares modulo every prime $p\le1000$, and compare with the bound of Theorem 8.10. Then test Theorem 8.8 on random coefficients at the Farey fractions of order $15$.

**P8.5 (★) CS application: the cost of a sieve.** Count the crossing-out operations of the sieve of Eratosthenes up to $n$ and compare with $n\log\log n$. Implement Euler's linear sieve, which crosses out each composite exactly once.

## Solutions

### P8.1

In [ ]:
import math, itertools, time
import numpy as np
import mpmath as mp

def primes_upto(n):
    s = np.ones(n + 1, dtype=bool); s[:2] = False
    for p in range(2, int(n**0.5) + 1):
        if s[p]: s[p*p::p] = False
    return s

def legendre_phi(x, z):
    ps = [p for p in range(2, z + 1) if all(p % q for q in range(2, int(p**0.5) + 1))]
    total, terms = 0, 0
    for k in range(len(ps) + 1):
        for c in itertools.combinations(ps, k):
            d = math.prod(c); total += (-1)**k * (x // d); terms += 1
    return total, terms

for x, z in ((1000, 31), (10**5, 13)):
    phi, terms = legendre_phi(x, z)
    direct = sum(1 for n in range(1, x + 1) if all(n % p for p in range(2, z + 1) if all(p % q for q in range(2, int(p**0.5) + 1))))
    prod = math.prod(1 - 1 / p for p in range(2, z + 1) if all(p % q for q in range(2, int(p**0.5) + 1)))
    print(f"x = {x}, z = {z}: Legendre sum = {phi} ({terms} terms), direct count = {direct}, main term x*prod = {x * prod:.1f}")

S = primes_upto(10**7)
P = np.nonzero(S)[0]
for k in range(3, 8):
    x = 10**k; z = int(math.isqrt(x))
    main = x * np.prod(1 - 1 / P[P <= z])
    phi = int(S[z + 1:x + 1].sum()) + 1
    print(f"x = 1e{k}: x*prod/Phi(x, sqrt x) = {main / phi:.4f}   (limit 2e^-gamma = {2 * math.exp(-float(mp.euler)):.4f})")

With $z$ small, Legendre's formula is exact and the main term is excellent, but the number of terms is $2^{\pi(z)}$: already $2048$ for $z=31$. At $z=\sqrt x$ the ratio drifts upward very slowly, $0.97$, $1.00$, $1.01$, $1.03$, $1.05$, towards the limit $1.1229$ predicted by Mertens and the prime number theorem. The approach is so slow because the lower-order terms decay like $\frac1{\log x}$. Nothing in the data would make you guess the limit: a good reason to prove things.

### P8.2

In [ ]:
def selberg_weights(y):
    # integers sieve: g(d) = 1/d, h(d) = 1/phi(d); lambda_d from the proof of the theorem
    sqf = [d for d in range(1, y + 1) if all(d % (p * p) for p in range(2, int(d**0.5) + 1))]
    def phi(n):
        r, m, p = n, n, 2
        while p * p <= m:
            if m % p == 0:
                while m % p == 0: m //= p
                r -= r // p
            p += 1
        if m > 1: r -= r // m
        return r
    def mob(n):
        r, m, p = 1, n, 2
        while p * p <= m:
            if m % p == 0:
                m //= p
                if m % p == 0: return 0
                r = -r
            p += 1
        return -r if m > 1 else r
    h = {d: 1 / phi(d) for d in sqf}
    G = sum(h.values())
    lam = {}
    for d in sqf:
        inner = sum(h[m] for m in sqf if m <= y // d and math.gcd(m, d) == 1)
        lam[d] = mob(d) * h[d] * d / G * inner
    return lam, G

y = 60
lam, G = selberg_weights(y)
print(f"y = {y}: G(y) = {G:.4f} >= log y = {math.log(y):.4f};  max |lambda_d| = {max(abs(v) for v in lam.values()):.4f}")
print("first weights:", {d: round(v, 3) for d, v in list(lam.items())[:10]})

x0, N = 10**9, 10**5
seg = np.ones(N, dtype=bool)                      # n = x0 + 1 + i
for p in P[P <= int(math.isqrt(x0 + N)) + 1]:
    start = (-(x0 + 1)) % p
    seg[start::p] = False
true_count = int(seg.sum())
ns = np.arange(x0 + 1, x0 + N + 1)
w = np.zeros(N)
for d, l in lam.items():
    w[(ns % d) == 0] += l
bound = float(np.sum(w**2))
print(f"primes in ]1e9, 1e9 + 1e5]: {true_count};  Selberg sum of squares with y = {y}: {bound:.0f};  main term N/G = {N / G:.0f};  2N/log N = {2 * N / math.log(N):.0f}")

The weights start at $\lambda_1=1$, alternate in sign like $\mu(d)$, and shrink in size as $d$ approaches $y$: a smoothed version of the Möbius function truncated at $y$. Brun's and Selberg's insight is that this smoothing is what keeps the error under control. The bound $18\,520$ is far above the true $4832$ because $y=60$ is tiny and $G(60)=5.4$ is a poor substitute for $\log N$; the asymptotic bound $\frac{2N}{\log N}$ needs $y$ near $\sqrt N$, and is still about $3.6$ times the truth here since the interval sits at height $10^9$, where the density of primes is $\frac1{\log10^9}$ rather than $\frac2{\log N}$. That factor is the price of uniformity: the same bound holds for every interval of length $N$, however far out.

### P8.3

In [ ]:
tw = P[S[np.minimum(P + 2, len(S) - 1)] & (P + 2 < len(S))]
C2 = float(np.prod(1 - 1 / (P[1:].astype(float) - 1)**2))
Li2 = lambda x: float(mp.quad(lambda t: 1 / mp.log(t)**2, [2, x]))
for k in (4, 5, 6, 7):
    x = 10**k
    print(f"x = 1e{k}: pi_2(x) = {np.sum(tw <= x):6d},  2 C_2 Li_2(x) = {2 * C2 * Li2(x):9.1f}")
print(f"C_2 = {C2:.7f}")
for k in (5, 6, 7):
    x = 10**k; t = tw[tw <= x].astype(float)
    part = float(np.sum(1 / t + 1 / (t + 2)))
    print(f"Brun partial sum to 1e{k}: {part:.7f};  plus 4 C_2 / log x: {part + 4 * C2 / math.log(x):.7f}")

The Hardy–Littlewood prediction is within $1\%$ of the truth everywhere, and Brun's series converges at a crawl: up to $10^7$ the partial sum is only $1.738$. Adding the predicted tail $\frac{4C_2}{\log x}$ gives $1.90219$, very close to the value $B_2=1.9021605\ldots$ obtained from twin primes up to $10^{16}$. Note that this uses a conjecture: nobody can prove that $B_2$ is not, say, $1.8$, since that would require a lower bound for twin primes.

### P8.4

In [ ]:
Nn, Q = 10**6, 1000
n = np.arange(1, Nn + 1); ok = np.ones(Nn, dtype=bool)
for p in P[P <= Q]:
    sq = np.zeros(p, dtype=bool); sq[(np.arange(p)**2) % p] = True
    ok &= sq[n % p]
survivors = n[ok]
print(f"n <= 1e6 that are squares mod every prime p <= 1000: {len(survivors)}; all perfect squares: {all(math.isqrt(int(v))**2 == v for v in survivors)}")
L = 1 + sum((p - 1) / (p + 1) for p in P[(P <= Q) & (P > 2)])
print(f"large sieve bound (pi N + Q^2) / L with L >= {L:.1f} (prime terms only): {(math.pi * Nn + Q * Q) / L:.0f}")

rng = np.random.default_rng(1)
M, N2, Q2 = 0, 200, 15
a = rng.normal(size=N2) + 1j * rng.normal(size=N2)
pts = [aa / q for q in range(1, Q2 + 1) for aa in range(1, q + 1) if math.gcd(aa, q) == 1]
lhs = sum(abs(np.sum(a * np.exp(2j * math.pi * np.arange(M + 1, M + N2 + 1) * t)))**2 for t in pts)
print(f"large sieve inequality, random a_n: LHS / sum|a_n|^2 = {lhs / np.sum(abs(a)**2):.1f};  pi N + Q^2 = {math.pi * N2 + Q2**2:.1f};  sharp N - 1 + Q^2 = {N2 - 1 + Q2**2}")

Only the $1000$ perfect squares survive, against a large sieve bound of about $25\,000$: the bound has the right shape ($\sqrt N\log N$ in general) but loses a logarithm and a constant. On random coefficients the large sieve inequality is far from tight; it is tight for coefficients that conspire with the points, like the indicator of a set that is unevenly spread in every residue class. Proving that nothing but squares (and quadratic-like sets) can survive is the open inverse problem mentioned in the chapter.

### P8.5

In [ ]:
def eratosthenes_ops(n):
    s = bytearray([1]) * (n + 1); ops = 0
    for p in range(2, math.isqrt(n) + 1):
        if s[p]:
            cnt = len(range(p * p, n + 1, p)); s[p*p::p] = bytes(cnt); ops += cnt
    return ops
def linear_sieve(n):
    lp = [0] * (n + 1); primes = []; ops = 0
    for i in range(2, n + 1):
        if lp[i] == 0: lp[i] = i; primes.append(i)
        for p in primes:
            if p > lp[i] or i * p > n: break
            lp[i * p] = p; ops += 1
    return primes, ops
for k in (4, 5, 6, 7):
    n = 10**k
    ops = eratosthenes_ops(n)
    print(f"n = 1e{k}: Eratosthenes crossings = {ops:9d},  n log log n = {n * math.log(math.log(n)):11.0f},  ratio = {ops / (n * math.log(math.log(n))):.3f}")
for k in (5, 6):
    n = 10**k
    t0 = time.perf_counter(); pr, ops = linear_sieve(n); t1 = time.perf_counter()
    print(f"n = 1e{k}: linear sieve writes = {ops} = n - pi(n) - 1 = {n - len(pr) - 1}  ({t1 - t0:.2f} s in pure Python)")

Crossing out the multiples of each prime $p\le\sqrt n$ starting at $p^2$ costs $\sum_{p\le\sqrt n}\frac{n-p^2}p\approx n\log\log\sqrt n=n(\log\log n-\log2)$ operations, by Mertens's second theorem (Theorem 1.5): the ratio tends to $1$, slowly, from below. Euler's linear sieve writes each composite $m$ once, from its smallest prime factor, so it does exactly $n-\pi(n)-1$ writes: linear time, at the price of storing the list of primes. In practice the segmented sieve of Eratosthenes wins, because its memory accesses stay in the processor cache; this is how all large prime tables, including those behind Brun's constant, are computed.